# DX 704 Week 2 Project

This week's project will analyze fresh strawberry price data for a hypothetical "buy low, freeze, and sell high" business.
Strawberries show strong seasonality in their prices compared to other fruits.

![](https://ers.usda.gov/sites/default/files/_laserfiche/Charts/61401/oct14_finding_plattner_fig01.png)

Image source: https://www.ers.usda.gov/amber-waves/2014/october/seasonal-fresh-fruit-price-patterns-differ-across-commodities-the-case-of-strawberries-and-apples

You are considering a business where you buy strawberries when the prices are very low, carefully freeze them, even more carefully defrost them, and then sell them when the prices are high.
You will forecast strawberry price time series and then use them to tactically pick times to buy, freeze, and sell the strawberries.

The full project description, a template notebook, and raw data are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-02


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Backtest Strawberry Prices

Read the provided "strawberry-prices.tsv" with data from 2020 through 2025.
This data is based on data from the U.S. Bureau of Statistics, but transformed so the ground truth is not online.
https://fred.stlouisfed.org/series/APU0000711415

Use the data for 2020 through 2024 to predict monthly prices in 2025.
Spend some time to make sure you are happy with your methodology and prediction accuracy, since you will reuse the methodology to forecast 2026 next.
Save the 2025 backtest predictions as "strawberry-backtest.tsv" with columns month and price.

In [ ]:
Hint: beware of missing rows of data.
The source is missing a few months!

In [2]:
%pip install pandas numpy matplotlib seaborn scikit-learn tensorflow

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow
Note: you may need to restart the kernel to use updated packages.


In [3]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the strawberry price data
df = pd.read_csv("strawberry-prices.tsv", sep="\t")

# Convert month to datetime
df["month"] = pd.to_datetime(df["month"])

# Sort by date
df = df.sort_values("month").reset_index(drop=True)

df.head()

,month,price
0,2020-01-01,4.049
1,2020-02-01,3.625
2,2020-03-01,3.377
3,2020-05-01,3.126
4,2020-06-01,2.926


In [5]:
# Create a complete monthly date range
all_months = pd.date_range(
    start=df["month"].min(),
    end=df["month"].max(),
    freq="MS"
)

missing_months = all_months.difference(df["month"])

print("Missing months:")
print(missing_months)

# Reindex so every month exists
complete_df = (
    df.set_index("month")
      .reindex(all_months)
      .rename_axis("month")
)

# Fill missing historical months using time interpolation
complete_df["price"] = complete_df["price"].interpolate(method="time")

complete_df = complete_df.reset_index()

complete_df.head(15)

train = complete_df[
    complete_df["month"] < "2025-01-01"
].copy()

test = df[
    (df["month"] >= "2025-01-01") &
    (df["month"] <= "2025-12-01")
].copy()

print("Training rows:", len(train))
print("Available actual 2025 rows:", len(test))

print(test)
from sklearn.linear_model import LinearRegression

# Time trend
train["t"] = np.arange(len(train))

# Month number for seasonality
train["month_num"] = train["month"].dt.month

# Create monthly dummy variables
X_train = pd.get_dummies(
    train[["t", "month_num"]],
    columns=["month_num"],
    drop_first=True
)

y_train = train["price"]

model = LinearRegression()
model.fit(X_train, y_train)
future_months = pd.date_range(
    start="2025-01-01",
    end="2025-12-01",
    freq="MS"
)

future = pd.DataFrame({
    "month": future_months
})

future["t"] = np.arange(
    len(train),
    len(train) + len(future)
)

future["month_num"] = future["month"].dt.month

X_future = pd.get_dummies(
    future[["t", "month_num"]],
    columns=["month_num"],
    drop_first=True
)

# Make sure future columns match training columns
X_future = X_future.reindex(
    columns=X_train.columns,
    fill_value=0
)

future["price"] = model.predict(X_future)

future[["month", "price"]]

strawberry_backtest = future[["month", "price"]].copy()

strawberry_backtest["month"] = (
    strawberry_backtest["month"]
    .dt.strftime("%Y-%m-01")
)

strawberry_backtest.to_csv(
    "strawberry-backtest.tsv",
    sep="\t",
    index=False
)

print(strawberry_backtest)
print("\nstrawberry-backtest.tsv saved successfully!")

Missing months:
DatetimeIndex(['2020-04-01', '2021-12-01', '2025-10-01', '2025-11-01'], dtype='datetime64[us]', freq=None)
Training rows: 60
Available actual 2025 rows: 10
        month  price
58 2025-01-01  4.584
59 2025-02-01  4.077
60 2025-03-01  3.369
61 2025-04-01  3.518
62 2025-05-01  3.416
63 2025-06-01  3.190
64 2025-07-01  3.187
65 2025-08-01  3.430
66 2025-09-01  3.645
67 2025-12-01  4.988
         month     price
0   2025-01-01  4.749648
1   2025-02-01  4.374048
2   2025-03-01  3.947848
3   2025-04-01  3.996736
4   2025-05-01  3.716848
5   2025-06-01  3.463848
6   2025-07-01  3.426848
7   2025-08-01  3.709648
8   2025-09-01  3.862248
9   2025-10-01  4.132648
10  2025-11-01  4.650648
11  2025-12-01  5.050890

strawberry-backtest.tsv saved successfully!


Please use the same format for the month column as in the training data, i.e. YYYY-MM-01.
The autograder may not be able to parse other formats.

Submit "strawberry-backtest.tsv" in Gradescope.

## Part 2: Backtest Errors

What are the mean and standard deviation of the residuals between your backtest predictions and the ground truth?

Write the mean and standard deviation to a file "backtest-accuracy.tsv" with two columns, mean and std.

In [6]:
# YOUR CHANGES HERE

comparison = test.merge(
    future[["month", "price"]],
    on="month",
    suffixes=("_actual", "_predicted")
)

comparison

,month,price_actual,price_predicted
0,2025-01-01,4.584,4.749648
1,2025-02-01,4.077,4.374048
2,2025-03-01,3.369,3.947848
3,2025-04-01,3.518,3.996736
4,2025-05-01,3.416,3.716848
5,2025-06-01,3.190,3.463848
6,2025-07-01,3.187,3.426848
7,2025-08-01,3.430,3.709648
8,2025-09-01,3.645,3.862248
9,2025-12-01,4.988,5.050890


In [7]:
comparison["residual"] = (
    comparison["price_actual"]
    - comparison["price_predicted"]
)

comparison[[
    "month",
    "price_actual",
    "price_predicted",
    "residual"
]]
residual_mean = comparison["residual"].mean()
residual_std = comparison["residual"].std()

print("Mean residual:", residual_mean)
print("Standard deviation of residuals:", residual_std)
backtest_accuracy = pd.DataFrame({
    "mean": [residual_mean],
    "std": [residual_std]
})

backtest_accuracy
backtest_accuracy.to_csv(
    "backtest-accuracy.tsv",
    sep="\t",
    index=False
)

print("backtest-accuracy.tsv saved successfully!")
check_accuracy = pd.read_csv(
    "backtest-accuracy.tsv",
    sep="\t"
)

print(check_accuracy)
print("Shape:", check_accuracy.shape)
print("Columns:", check_accuracy.columns.tolist())

Mean residual: -0.2894606557377043
Standard deviation of residuals: 0.14698348694740387
backtest-accuracy.tsv saved successfully!
       mean       std
0 -0.289461  0.146983
Shape: (1, 2)
Columns: ['mean', 'std']


In [9]:
residual_mean = comparison["residual"].mean()
residual_std = comparison["residual"].std()

print("Mean residual:", residual_mean)
print("Standard deviation of residuals:", residual_std)
backtest_accuracy = pd.DataFrame({
    "mean": [residual_mean],
    "std": [residual_std]
})

backtest_accuracy
backtest_accuracy.to_csv(
    "backtest-accuracy.tsv",
    sep="\t",
    index=False
)

print("backtest-accuracy.tsv saved successfully!")
check_accuracy = pd.read_csv(
    "backtest-accuracy.tsv",
    sep="\t"
)

print(check_accuracy)
print("Shape:", check_accuracy.shape)
print("Columns:", check_accuracy.columns.tolist())

Mean residual: -0.2894606557377043
Standard deviation of residuals: 0.14698348694740387
backtest-accuracy.tsv saved successfully!
       mean       std
0 -0.289461  0.146983
Shape: (1, 2)
Columns: ['mean', 'std']


Hint: If the mean residual in your backtest is not close to zero, then your model is likely missing a systematic change and you should go back to improve it.

Submit "backtest-accuracy.tsv" in Gradescope.

## Part 3: Forecast Strawberry Prices

Use all the data from 2020 through 2025 to predict monthly prices in 2026 using the same methodology from part 1.
Make a monthly forecast for each month of 2026 and save it as "strawberry-forecast.tsv" with columns for month and price.


In [10]:
# YOUR CHANGES HERE
full_data = complete_df[
    complete_df["month"] <= "2025-12-01"
].copy()

# Create time trend and month number
full_data["t"] = np.arange(len(full_data))
full_data["month_num"] = full_data["month"].dt.month

# Create seasonal dummy variables
X_full = pd.get_dummies(
    full_data[["t", "month_num"]],
    columns=["month_num"],
    drop_first=True
)

y_full = full_data["price"]

# Fit the same regression model used in Part 1
forecast_model = LinearRegression()
forecast_model.fit(X_full, y_full)
forecast_months = pd.date_range(
    start="2026-01-01",
    end="2026-12-01",
    freq="MS"
)

forecast_2026 = pd.DataFrame({
    "month": forecast_months
})

forecast_2026["t"] = np.arange(
    len(full_data),
    len(full_data) + 12
)

forecast_2026["month_num"] = forecast_2026["month"].dt.month
X_2026 = pd.get_dummies(
    forecast_2026[["t", "month_num"]],
    columns=["month_num"],
    drop_first=True
)

# Match the training columns exactly
X_2026 = X_2026.reindex(
    columns=X_full.columns,
    fill_value=0
)
forecast_2026["price"] = forecast_model.predict(X_2026)

forecast_2026[["month", "price"]]
strawberry_forecast = forecast_2026[
    ["month", "price"]
].copy()

strawberry_forecast["month"] = (
    strawberry_forecast["month"]
    .dt.strftime("%Y-%m-01")
)

strawberry_forecast.to_csv(
    "strawberry-forecast.tsv",
    sep="\t",
    index=False
)

print(strawberry_forecast)
print("\nstrawberry-forecast.tsv saved successfully!")
check_forecast = pd.read_csv(
    "strawberry-forecast.tsv",
    sep="\t"
)

print(check_forecast)
print("Shape:", check_forecast.shape)
print("Columns:", check_forecast.columns.tolist())


         month     price
0   2026-01-01  4.677985
1   2026-02-01  4.280485
2   2026-03-01  3.807318
3   2026-04-01  3.872892
4   2026-05-01  3.622651
5   2026-06-01  3.374151
6   2026-07-01  3.342818
7   2026-08-01  3.618985
8   2026-09-01  3.781985
9   2026-10-01  4.081109
10  2026-11-01  4.589027
11  2026-12-01  4.996353

strawberry-forecast.tsv saved successfully!
         month     price
0   2026-01-01  4.677985
1   2026-02-01  4.280485
2   2026-03-01  3.807318
3   2026-04-01  3.872892
4   2026-05-01  3.622651
5   2026-06-01  3.374151
6   2026-07-01  3.342818
7   2026-08-01  3.618985
8   2026-09-01  3.781985
9   2026-10-01  4.081109
10  2026-11-01  4.589027
11  2026-12-01  4.996353
Shape: (12, 2)
Columns: ['month', 'price']


Submit "strawberry-forecast.tsv" in Gradescope.

## Part 4: Buy Low, Freeze and Sell High

Using your 2026 forecast, analyze the profit picking different pairs of months to buy and sell strawberries.
Maximize your profit assuming that it costs &dollar;0.20 per pint to freeze the strawberries, &dollar;0.10 per pint per month to store the frozen strawberries and there is a 10% price discount from selling previously frozen strawberries.
So, if you buy a pint of strawberies for &dollar;1, freeze them, and sell them for &dollar;2 three months after buying them, then the profit is &dollar;2 * 0.9 - &dollar;1 - &dollar;0.20 - &dollar;0.10 * 3 = &dollar;0.30 per pint.
To evaluate a given pair of months, assume that you can invest &dollar;1,000,000 to cover all costs, and that you buy as many pints of strawberries as possible.

Write the results of your analysis to a file "timings.tsv" with columns for the buy_month, sell_month, pints_purchased, and expected_profit.

In [11]:
# YOUR CHANGES HERE
prices_2026 = forecast_2026[["month", "price"]].copy()

prices_2026


,month,price
0,2026-01-01,4.677985
1,2026-02-01,4.280485
2,2026-03-01,3.807318
3,2026-04-01,3.872892
4,2026-05-01,3.622651
5,2026-06-01,3.374151
6,2026-07-01,3.342818
7,2026-08-01,3.618985
8,2026-09-01,3.781985
9,2026-10-01,4.081109


In [12]:
budget = 1_000_000
storage_cost_per_month = 0.10
frozen_discount = 0.10

timing_results = []

for buy_i in range(len(prices_2026)):
    for sell_i in range(buy_i + 1, len(prices_2026)):

        buy_month = prices_2026.iloc[buy_i]["month"]
        sell_month = prices_2026.iloc[sell_i]["month"]

        buy_price = prices_2026.iloc[buy_i]["price"]
        sell_price = prices_2026.iloc[sell_i]["price"]

        # Number of months strawberries are stored
        months_stored = sell_i - buy_i

        # Storage cost per pint
        storage_cost = storage_cost_per_month * months_stored

        # Total cost per pint
        total_cost_per_pint = buy_price + storage_cost

        # Buy as many whole pints as possible
        pints_purchased = int(budget // total_cost_per_pint)

        # Frozen strawberries sell at a 10% discount
        discounted_sell_price = sell_price * (1 - frozen_discount)

        # Profit per pint
        profit_per_pint = (
            discounted_sell_price
            - buy_price
            - storage_cost
        )

        # Total expected profit
        expected_profit = pints_purchased * profit_per_pint

        timing_results.append({
            "buy_month": buy_month,
            "sell_month": sell_month,
            "pints_purchased": pints_purchased,
            "expected_profit": expected_profit
        })

timings = pd.DataFrame(timing_results)

timings
best_strategy = timings.loc[
    timings["expected_profit"].idxmax()
]

print(best_strategy)
timings["buy_month"] = (
    pd.to_datetime(timings["buy_month"])
    .dt.strftime("%Y-%m-01")
)

timings["sell_month"] = (
    pd.to_datetime(timings["sell_month"])
    .dt.strftime("%Y-%m-01")
)
timings.to_csv(
    "timings.tsv",
    sep="\t",
    index=False
)

print("timings.tsv saved successfully!")
check_timings = pd.read_csv(
    "timings.tsv",
    sep="\t"
)

print(check_timings.head())
print("\nShape:", check_timings.shape)
print("Columns:", check_timings.columns.tolist())

buy_month          2026-07-01 00:00:00
sell_month         2026-12-01 00:00:00
pints_purchased                 260225
expected_profit          170161.173715
Name: 55, dtype: object
timings.tsv saved successfully!
    buy_month  sell_month  pints_purchased  expected_profit
0  2026-01-01  2026-02-01           209293   -193710.813061
1  2026-01-01  2026-03-01           205002   -297539.586335
2  2026-01-01  2026-04-01           200884   -299795.678312
3  2026-01-01  2026-05-01           196928   -357936.028799
4  2026-01-01  2026-06-01           193125   -413528.608120

Shape: (66, 4)
Columns: ['buy_month', 'sell_month', 'pints_purchased', 'expected_profit']


Submit "timings.tsv" in Gradescope.

## Part 5: Strategy Check

What is the best profit scenario according to your previous timing analysis?
How much does that profit change if the sell price is off by one standard deviation from your backtest analysis?
(Variation in the sell price is more dangerous because you can see the buy price before fully committing.)

Write the results to a file "check.tsv" with columns `best_profit` and `one_std_profit`.
To be clear, `one_std_profit` should be the number of pints bought in your best profit scenario times your backtested standard deviation of the residual.
This represents the standard deviation in revenue when selling if you explicitly assume that you buy according to the best profit scenario and your backtest standard deviation is representative of the future prices.

In [13]:
# YOUR CHANGES HERE

best_strategy = timings.loc[
    timings["expected_profit"].idxmax()
]

best_profit = best_strategy["expected_profit"]
best_pints = best_strategy["pints_purchased"]

print("Best profit:", best_profit)
print("Pints purchased:", best_pints)

one_std_profit = best_pints * residual_std

print("One standard deviation profit change:", one_std_profit)

check = pd.DataFrame({
    "best_profit": [best_profit],
    "one_std_profit": [one_std_profit]
})

check
check.to_csv(
    "check.tsv",
    sep="\t",
    index=False
)

print("check.tsv saved successfully!")
check_file = pd.read_csv(
    "check.tsv",
    sep="\t"
)

print(check_file)
print("\nShape:", check_file.shape)
print("Columns:", check_file.columns.tolist())

Best profit: 170161.17371482257
Pints purchased: 260225
One standard deviation profit change: 38248.777890888174
check.tsv saved successfully!
     best_profit  one_std_profit
0  170161.173715    38248.777891

Shape: (1, 2)
Columns: ['best_profit', 'one_std_profit']


Submit "check.tsv" in Gradescope.

## Part 6: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

In [14]:
acknowledgment_text = """I used Python libraries including pandas, NumPy, matplotlib, and scikit-learn for data processing, analysis, forecasting, and visualization. I used ChatGPT for assistance with understanding the project instructions, developing and troubleshooting Python code, and reviewing the required output file formats. I reviewed and executed the calculations and results in my Jupyter Notebook.


"""

with open("acknowledgments.txt", "w") as file:
    file.write(acknowledgment_text)

print("acknowledgments.txt saved successfully!")

acknowledgments.txt saved successfully!


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.

In [15]:
import os

required_files = [
    "project.ipynb",
    "strawberry-backtest.tsv",
    "backtest-accuracy.tsv",
    "strawberry-forecast.tsv",
    "timings.tsv",
    "check.tsv",
    "acknowledgments.txt"
]

print("FINAL PROJECT FILE CHECK")
print("-" * 35)

for file in required_files:
    if os.path.exists(file):
        print(f"{file}: ✓")
    else:
        print(f"{file}: MISSING")

FINAL PROJECT FILE CHECK
-----------------------------------
project.ipynb: MISSING
strawberry-backtest.tsv: ✓
backtest-accuracy.tsv: ✓
strawberry-forecast.tsv: ✓
timings.tsv: ✓
check.tsv: ✓
acknowledgments.txt: ✓
